# 04 - Deep Learning Models

Train LSTM and TCN on sequence data.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import polars as pl
import torch
from sklearn.preprocessing import StandardScaler

from src.models.lstm_model import LSTMRegressor, prepare_sequences
from src.models.tcn_model import TCNRegressor
from src.models.train import train_torch_model, evaluate_model
from src.evaluation.metrics import comparison_table

In [ ]:
df_feat = pl.read_parquet('../data/processed/hourly_demand.parquet')
feature_cols = [c for c in df_feat.columns if c not in ('pickup_hour', 'PULocationID', 'demand', 'pickup_zone', 'borough')]
df_feat = df_feat.drop_nulls().sort('pickup_hour')
n = df_feat.height
X = df_feat.select(feature_cols).to_numpy()
y = df_feat.select('demand').to_numpy().ravel()

train_end, val_end = int(n * 0.7), int(n * 0.85)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X[:train_end])
X_val_s = scaler.transform(X[train_end:val_end])
X_test_s = scaler.transform(X[val_end:])
y_train, y_val, y_test = y[:train_end], y[train_end:val_end], y[val_end:]

In [ ]:
seq_len = 24
X_train_seq, y_train_seq = prepare_sequences(torch.tensor(np.column_stack([X_train_s, y_train]), dtype=torch.float32), seq_len)
X_val_seq, y_val_seq = prepare_sequences(torch.tensor(np.column_stack([X_val_s, y_val]), dtype=torch.float32), seq_len)
X_test_seq, y_test_seq = prepare_sequences(torch.tensor(np.column_stack([X_test_s, y_test]), dtype=torch.float32), seq_len)

In [ ]:
lstm = LSTMRegressor(input_size=X_train_s.shape[1])
lstm = train_torch_model(lstm, X_train_seq, y_train_seq, X_val_seq, y_val_seq, epochs=30)

In [ ]:
tcn = TCNRegressor(input_size=X_train_s.shape[1])
tcn = train_torch_model(tcn, X_train_seq, y_train_seq, X_val_seq, y_val_seq, epochs=30)

In [ ]:
results = {
    'LSTM': evaluate_model(lstm, X_test_seq, y_test_seq),
    'TCN': evaluate_model(tcn, X_test_seq, y_test_seq),
}
comparison_table(results)